In [1]:
%matplotlib widget

In [2]:
from glob import glob
import numpy as np
import pandas as pd
import flammkuchen as fl
from split_dataset import SplitDataset
from bouter import Experiment
from fimpy.pipeline.general import calc_f0, dff
from motions.utilities import stim_vel_dir_dataframe, quantize_directions, stim_vel_dir_dataframe_from_stimlog
from scipy.interpolate import interp1d 
from scipy.signal import convolve2d
import colorspacious
import napari
import matplotlib.pyplot as plt
import json

from fimpylab.core.twop_experiment import TwoPExperiment

from pathlib import Path

In [3]:
# make sensory regressors. requires old bouter stimulus_param_log.

def make_sensory_regressors(stim_log, n_dirs=8, upsampling=5, sampling=1/3):
    stim = stim_vel_dir_dataframe_from_stimlog(stim_log)
    bin_centres, dir_bins = quantize_directions(stim.theta)
    ind_regs = np.zeros((n_dirs, len(stim)))
    for i_dir in range(n_dirs):
        ind_regs[i_dir, :] = (np.abs(dir_bins - i_dir) < 0.1) & (stim.vel > 0.1)  

    dt_upsampled = sampling / upsampling
        
    t_imaging_up = np.arange(0, stim.t.values[-1], dt_upsampled)
    reg_up = interp1d(stim.t.values, ind_regs, axis=1, fill_value="extrapolate")(
        t_imaging_up
    )
    
    # 6s kernel
    u_steps = t_imaging_up.shape[0]
    print(stim.t.values[-1])
    u_time = np.arange(u_steps) * dt_upsampled
    decay = np.exp(-u_time / (1.5 / np.log(2)))
    kernel = decay / np.sum(decay)
    
    convolved = convolve2d(reg_up, kernel[None, :])[:, 0:u_steps]
    reg_sensory = convolved[:, ::upsampling]
    reg_sensory = reg_sensory[:, :len_rec]
    reg_up = reg_up[:, ::upsampling]

    return pd.DataFrame(reg_sensory.T, columns=[f"motion_{i}" for i in range(n_dirs)]), reg_up


In [4]:
# find the frames to calculate the baseline.
def no_regressor_frames(regressors, threshold=0.01):
    return np.where(np.all(regressors.values < threshold, axis=1))[0]

# calculate the baseline, plane-wise
def calc_f0(stack, frames):
    fr_mean = None
    for i_frame in frames:
        sf = stack[int(i_frame), :, :]
        if fr_mean is None:
            fr_mean = sf
        else:
            fr_mean += sf
    return fr_mean / len(frames)

In [5]:
master =  Path(r"Z:\Hagar\vision and navigation - motion\v10\gad1b new")
fish_list = list(master.glob("*_gad1b*"))

n_dirs = 8

In [6]:
fish_list

[WindowsPath('Z:/Hagar/vision and navigation - motion/v10/gad1b new/241001_f0_gad1b'),
 WindowsPath('Z:/Hagar/vision and navigation - motion/v10/gad1b new/241001_f1_gad1b'),
 WindowsPath('Z:/Hagar/vision and navigation - motion/v10/gad1b new/241001_f3_gad1b'),
 WindowsPath('Z:/Hagar/vision and navigation - motion/v10/gad1b new/241120_f1_gad1b')]

In [7]:
for fish in fish_list[0:]:
    print(fish)
    #try:
    if not (fish / "sensory_regressors.h5").exists():
        stack_file = str(fish / 'original' / "stack_metadata.json")
        exp_list = glob(str(fish / "*behavior*"))

        with open(stack_file) as f:
                stack_metadata = json.load(f)

        len_rec = stack_metadata["shape_full"][0]
        n_planes = stack_metadata["shape_full"][1]
        print(n_planes)

        exp_2p = TwoPExperiment(fish)
        sampling = exp_2p.dt_imaging
        time = np.linspace(0, len_rec*sampling, len_rec)
        print(len_rec)
        for plane in range(n_planes):

            tmp_stimlog = exp_2p.load_session_log('stimulus_log', plane)
            stim = stim_vel_dir_dataframe_from_stimlog(tmp_stimlog)

            theta = np.asarray(stim.theta)

            bin_centres, dir_bins = quantize_directions(stim.theta)
            ind_regs = np.zeros((n_dirs, len(stim)))
            for i_dir in range(n_dirs):
                ind_regs[i_dir, :] = (np.abs(dir_bins - i_dir) < 0.1) & (stim.vel > 0.1) 

            #len_rec, num_cells = np.shape(traces)
            # make a list of sensory regressors 
            reg, reg_interp = make_sensory_regressors(tmp_stimlog, sampling=sampling)
            reg_list = [reg]
            print(np.shape(reg))
            #print(len_rec)

            d = {
                'regressors': reg,
                'theta': theta,
                'individual_theta': ind_regs,
                'individual_theta_interp': reg_interp,
            }
            file_name = 'sensory_regressors' + str(plane) + '.h5'
            fl.save(fish / file_name, d)
    #except:
    #    print("Error")

Z:\Hagar\vision and navigation - motion\v10\gad1b new\241001_f0_gad1b
10
2411
799.998842
(2400, 8)
799.989542
(2400, 8)
799.983095
(2400, 8)
799.993502
(2400, 8)
799.981228
(2400, 8)
799.994511
(2400, 8)
799.992105
(2400, 8)
799.997089
(2400, 8)
799.992715
(2400, 8)
799.998619
(2400, 8)
Z:\Hagar\vision and navigation - motion\v10\gad1b new\241001_f1_gad1b
7
2411
799.990904
(2400, 8)
799.991904
(2400, 8)
799.993224
(2400, 8)
799.997003
(2400, 8)
799.985066
(2400, 8)
799.963803
(2400, 8)
799.979771
(2400, 8)
Z:\Hagar\vision and navigation - motion\v10\gad1b new\241001_f3_gad1b
7
2411
799.994242
(2400, 8)
799.989888
(2400, 8)
799.99755
(2400, 8)
799.998694
(2400, 8)
799.992472
(2400, 8)
799.999828
(2400, 8)
799.997567
(2400, 8)
Z:\Hagar\vision and navigation - motion\v10\gad1b new\241120_f1_gad1b
6
2411
799.996286
(2400, 8)
799.994562
(2400, 8)
799.984331
(2400, 8)
799.997838
(2400, 8)
799.990656
(2400, 8)
799.984869
(2400, 8)


In [ ]:
765/2306

In [ ]:
np.shape(theta)[0]/2306